## Check the Noise Floor
This notebook identifies the noise floor for our sample data since the data will naturally fluctuate due to normal variance or measurement errors. This offers a baseline so we can avoid false positive drifts and hopefully separate a real drift signal from the noise floor. 

**Method:**
We'll score two random halves of the same-issue week using the Unsupervised method's scoring steps. We use the same week because the same week = no change, so whatever change score we get from that we'll consider as our noise floor. 

**Output:** 
noise_floor.csv: one row per issue-week with its noise floor. It's NaN when the week has less than 100 tweets.

In [ ]:
import numpy as np
import pandas as pd
import time
import warnings
import sys
from sklearn.exceptions import ConvergenceWarning

sys.path.append("../CodeFiles")

from shift_scoring import (
    K, ISSUE_KEYWORDS, tag_issues, filter_issue_tweets, make_issue_vectorizer,
    fit_frames, shift_between, run_issue,
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
df = pd.read_parquet("../sampleddata.parquet")
df = tag_issues(df)
features = pd.read_csv("../DataFiles/tk_issue_week_features.csv")
print(df.shape)

In [ ]:
# Check A: my issue tweet counts should match Unsupervised table
my_counts = df[[f"issue_{i}" for i in ISSUE_KEYWORDS]].sum().rename(lambda c: c.removeprefix("issue_"))
her_counts = features.groupby("issue")["n_tweets_issue"].sum()
print(pd.DataFrame({"mine": my_counts, "hers": her_counts}))

In [ ]:
# Check B: the shared file should reproduce Unsupervised shift scores for every issue
for issue in ISSUE_KEYWORDS:
    mine = run_issue(df, issue)
    hers = features[features["issue"] == issue][["week", "shift_score"]].dropna()
    compare = mine.merge(hers, left_on="period", right_on="week", suffixes=("_mine", "_hers"))
    gap = (compare["shift_score_mine"] - compare["shift_score_hers"]).abs().max()
    print(f"{issue}: {len(compare)} weeks compared, biggest difference = {gap}")

##### Make sure shift_scoring.py reproduces Unsupervised shift scores

In [ ]:
mine = run_issue(df, "immigration")
hers = features[features["issue"] == "immigration"][["week", "shift_score"]]
compare = mine.merge(hers, left_on="period", right_on="week", suffixes=("_mine", "_hers"))
print((compare["shift_score_mine"] - compare["shift_score_hers"]).abs().max())

##### Make sure my keyword matching is same as Unsupervised feature table

In [ ]:
# If my keyword guess matches her changes. I can delete once she pushes her notebook.

mine = df[[f"issue_{i}" for i in ISSUE_KEYWORDS]].sum().rename(lambda c: c.removeprefix("issue_"))
hers = features.groupby("issue")["n_tweets_issue"].sum()

# Building the DataFrame first lines the rows up by issue name,
# so the comparison works even though the two lists were in different orders
check = pd.DataFrame({"mine": mine, "hers": hers})
check["match"] = check["mine"] == check["hers"]
print(check)

In [ ]:
def noise_floor(issue, week, k=K, n_repeats=20, seed=42):
    """Score two random halves of the same issue-week with Tanisha's scoring steps.
    Same week = no real change, so the score is chance alone (the noise floor)."""
    rng = np.random.default_rng(seed)

    # Same tweet filter and vocab as run_issue (shared in shift_scoring.py)
    tw = filter_issue_tweets(df, issue)
    vec = make_issue_vectorizer().fit(tw["cleanedTweet"])
    X_week = vec.transform(tw.loc[tw["week"] == week, "cleanedTweet"])

    # Each half needs 50+ tweets, so the week needs 100+ (same per-group rule as run_issue)
    n = X_week.shape[0]
    if n < 2 * k * 10:
        return np.nan

    # I'll average over many random splits for a more stable result
    scores = []
    for _ in range(n_repeats):
        idx = rng.permutation(n)
        A = fit_frames(X_week[idx[: n // 2]], k)
        B = fit_frames(X_week[idx[n // 2 :]], k)
        scores.append(shift_between(A, B)["shift_score"])
    return np.mean(scores)

##### Test on one week

In [ ]:
# Test on week 12
start = time.time()
test_noise = noise_floor("immigration", 12, n_repeats=3)
elapsed = time.time() - start

# The week-to-week shift score for the same issue-week
real_score = features.loc[
    (features["issue"] == "immigration") & (features["week"] == 12), "shift_score"
].iloc[0]

print(f"Noise floor (same week, two halves): {test_noise:.3f}")
print(f"Shift score (week 11 to 12):    {real_score:.3f}")

In [ ]:
# Use the feature table as the list of issue-weeks to score,
# so the noise floor lines up with the Unsupervised shift scores

rows = []
for issue, week in features[["issue", "week"]].itertuples(index=False):
    rows.append({
        "issue": issue,
        "week": week,
        "noise_floor": noise_floor(issue, week),   # NaN if too few tweets
    })

noise_df = pd.DataFrame(rows)
noise_df.to_csv("../DataFiles/noise_floor.csv", index=False)  # index=False skips the row-number column
noise_df.head()

**Summary:** For each issue, how many weeks have a noise floor and how often is the shift score higher than the noise floor. 

In [ ]:
# Join noise floors onto the real shift scores, row for row
check_nf = features[["issue", "week", "n_tweets_issue", "shift_score"]].merge(
    noise_df, on=["issue", "week"]
)

# Real change above the noise floor (negative = real score is within chance)
check_nf["shift_minus_noise"] = check_nf["shift_score"] - check_nf["noise_floor"]

# Only compare weeks that have BOTH a shift score and a noise floor,
# so the two averages describe the same weeks
both = check_nf.dropna(subset=["shift_score", "noise_floor"])

summary = check_nf.groupby("issue").agg(
    weeks=("week", "size"),
    weeks_with_noise_floor=("noise_floor", lambda s: s.notna().sum()),
    avg_tweets=("n_tweets_issue", "mean"),
).join(
    both.groupby("issue").agg(
        weeks_compared=("week", "size"),
        avg_shift=("shift_score", "mean"),
        avg_noise=("noise_floor", "mean"),
        # share of compared weeks where the real change beats chance
        pct_above_noise=("shift_minus_noise", lambda s: (s > 0).mean()),
    )
).round(3)
print(summary)

### Are there any clearly above-chance weeks?

In [ ]:
# Count weeks where the real shift beats chance by 0.05 or more
margin = 0.05
check_nf["clearly_above"] = check_nf["shift_minus_noise"] >= margin

print(check_nf.groupby("issue")["clearly_above"].sum())

# List those weeks so we can check them against real-world events
print(check_nf[check_nf["clearly_above"]][["issue", "week", "shift_score", "noise_floor"]]
      .sort_values("shift_score", ascending=False))

### What the results say

We compared only weeks that have both a shift score and a noise floor. For most issues, the average week-to-week shift score is about the same size as the noise floor. Economy averages 0.274 for real shifts and 0.289 for noise; democracy and immigration look similar. Foreign policy is the only issue whose average shift is above its noise (0.302 vs. 0.273). Real scores are higher than chance in 37–57% of weeks, depending on the issue.

Abortion is the weakest issue. Only 15 of 31 weeks had enough tweets (100+ issue tweets with 5+ words) to measure a noise floor. On the 14 weeks we could compare, its average shift (0.366) matches its noise (0.367). Its high overall shift score comes mostly from its smallest weeks, which is small-sample noise.

Each half has only half of the week's tweets, so this noise floor runs a little high. A negative shift_minus_noise means "not clearly above chance".

### What this means for the project

Narrative shifts shouldn't occur that often. In most weeks nothing new happens, so we expect most weeks to be close to the noise floor. Using a margin of 0.05, 30 issue-weeks are clearly above chance, 11 of which are in foreign policy.

The shift score can't be used on its own as a drift signal, so we'll use shift_minus_noise as a feature. Labels will come from the tweet text (LLM labels, hand-checked on a sample), not from these scores. The supervised model then tests whether the noisy scores have a useful signal when checked against those labels. Abortion should be dropped or marked low-confidence.

### Why it matters

Without this check, we could report ordinary week-to-week fluctuations as narrative shifts (false positives). We corrected for noise before modelinge because most change scores were near chance level.

### These results can be used to see if they line up with real events.

### LLM Prompt Building
This next cell is the first step for Supervised Labeling. I am building an LLM Prompt and need to provide 3 examples to the llm since they learn best from examples, so I'm using these results to help me more quickly find examples rather than me randomly looking through all the weeks myself. Looking for one example of each: shift_candidates, no_change_candidates

In [ ]:
# Shortlist candidate rows for the 3 prompt examples.

# I'll leave out abortion since its scores are mostly small-sample noise.
# I'll leave out weeks next to a collection gap (weeks 12, 13, 16, 17 have only
# 2-3 days of tweets), because an example should be a clean and typical case.
gap_weeks = [12, 13, 16, 17]
near_gap = check_nf["week"].isin(gap_weeks) | (check_nf["week"] - 1).isin(gap_weeks)
clean = check_nf[(check_nf["issue"] != "abortion") & ~near_gap]

# Shift candidates: the real change is greater than chance by the most
shift_candidates = clean[clean["clearly_above"]].sort_values("shift_minus_noise", ascending=False)

# "No change" candidates: the real change is well below chance
no_change_candidates = clean.sort_values("shift_minus_noise").head(8)

# Look up each candidate's row_id (the packet's file name)
key = pd.read_csv("../DataFiles/answer_key.csv")
cols = ["row_id", "issue", "week", "shift_minus_noise"]
print("Shift candidates (could be content or narrative shifts):")
print(shift_candidates.merge(key, on=["issue", "week"])[cols].to_string(index=False))
print("\nNo-change candidates:")
print(no_change_candidates.merge(key, on=["issue", "week"])[cols].to_string(index=False))